In [ ]:
import json
from concurrent.futures import ThreadPoolExecutor, as_completed

import boto3
from botocore import UNSIGNED
from botocore.config import Config
import pandas as pd
from web3 import Web3

import plotly.express as px

from mainnet_launch.database.schema.full import RebalancePlans, Destinations, DexSwapSteps, Tokens, DestinationStates

from mainnet_launch.database.schema.postgres_operations import (
    get_full_table_as_orm,
    insert_avoid_conflicts,
    get_subset_not_already_in_column,
)

from mainnet_launch.constants import ALL_AUTOPOOLS, AutopoolConstants, USDC, WETH, AUTO_USD
from mainnet_launch.data_fetching.block_timestamp import _fetch_block_df_from_subgraph


def convert_rebalance_plan_json_to_rebalance_plan_line(
    rebalance_plan_json_key: str, s3_client, autopool: AutopoolConstants
):
    plan = json.loads(
        s3_client.get_object(
            Bucket=autopool.solver_rebalance_plans_bucket,
            Key=rebalance_plan_json_key,
        )["Body"].read()
    )

    plan["rebalance_plan_json_key"] = rebalance_plan_json_key
    plan["autopool_vault_address"] = autopool.autopool_eth_addr
    return plan



def dicts_to_destination_states(plan: dict, autopool:AutopoolConstants, timestamp_to_block:dict[int, int], tokens_address_to_decimals:dict[str,int]) -> list[DestinationStates]:
    """
    Convert each dict in `items` into a DestinationStates ORM object,
    using direct lookups and computing fee_plus_base_apr as
    total_apr_out - incentive_apr. All numeric fields are cast to float.
    """

    state_of_destinations: list[DestinationStates] = []
    for dest_state in plan['sod']["destStates"]:
        # direct lookups and float conversion
        incentive = float(dest_state["incentiveAPR"])
        total_in = float(dest_state["totalAprIn"])
        total_out = float(dest_state["totalAprOut"])

        raw_total_supply = float(dest_state["totSupply"])
        normalized_total_supply = raw_total_supply / (10 ** tokens_address_to_decimals[dest_state["underlying"]])

        block = timestamp_to_block.get(plan['sod']['currentTimestamp'], None)
        state = DestinationStates(
            destination_vault_address=Web3.toChecksumAddress(dest_state["address"]),
            block=block,
            chain_id=autopool.chain.chain_id,
            incentive_apr=incentive,
            fee_apr=None,
            base_apr=None,
            points_apr=None,
            fee_plus_base_apr=total_out - (incentive / .9), # remove downscaling
            total_apr_in=total_in,
            total_apr_out=total_out,
            underlying_token_total_supply=normalized_total_supply,
            safe_total_supply=None,
            lp_token_spot_price=float(dest_state["spotPrice"]),
            lp_token_safe_price=float(dest_state["safePrice"]),
        )
        state_of_destinations.append(state)
    return state_of_destinations


s3_client = boto3.client("s3", config=Config(signature_version=UNSIGNED))

for autopool in [AUTO_USD]:

    solver_plan_paths_on_remote = [
        r["Key"] for r in s3_client.list_objects_v2(Bucket=autopool.solver_rebalance_plans_bucket).get("Contents")
    ]
    plan_timestamps = [int(p.split(('_'))[2]) for p in solver_plan_paths_on_remote]

    block_df = _fetch_block_df_from_subgraph(autopool.chain,plan_timestamps)
    timestamp_to_block = {int(t): b for t, b in zip(block_df["timestamp"], block_df["block"])}
    tokens_orm = get_full_table_as_orm(Tokens, where_clause=(Tokens.chain_id == autopool.chain.chain_id))
    tokens_address_to_decimals = {t.token_address: t.decimals for t in tokens_orm}

    def _process_plan(plan_path):
        plan = convert_rebalance_plan_json_to_rebalance_plan_line(
            plan_path, s3_client, autopool
        )
        return dicts_to_destination_states(
            plan, autopool, timestamp_to_block, tokens_address_to_decimals
        )

    all_destination_states = []
    with ThreadPoolExecutor(max_workers=8) as executor:
        # submit one future per plan_path
        futures = {executor.submit(_process_plan, path): path for path in solver_plan_paths_on_remote}

        for fut in as_completed(futures):
            try:
                states = fut.result()
                all_destination_states.extend(states)
                print(len(all_destination_states))
            except Exception as e:
                print(f"Error processing {futures[fut]}: {e}")


2025-05-09 11:44:32,822 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-05-09 11:44:32,823 INFO sqlalchemy.engine.Engine 
            SELECT *
            FROM tokens
            WHERE tokens.chain_id = 1
        
2025-05-09 11:44:32,824 INFO sqlalchemy.engine.Engine [cached since 113.9s ago] {}
2025-05-09 11:44:33,030 INFO sqlalchemy.engine.Engine COMMIT


KeyboardInterrupt: 

In [ ]:
_fetch_block_df_from_subgraph(autopool.chain,[1743377226])

ValueError: number sections must be larger than 0.

KeyError: 1743377226

In [ ]:
plan

{'timestamp': 1743377226,
 'sodOnly': False,
 'chainId': '1',
 'solverAddress': '0xD02b50CFc6c2903bF13638B28D081ad11515B6f9',
 'poolAddress': '0xa7569A44f348d3D70d8ad5889e50F78E33d80D35',
 'destinationOut': '0xa7569A44f348d3D70d8ad5889e50F78E33d80D35',
 'tokenOut': '0xA0b86991c6218b36c1d19D4a2e9Eb0cE3606eB48',
 'amountOut': '100000000',
 'amountOutUSD': '100000000',
 'destinationIn': '0x6F628dcCD275feA4277722D177265741031E09d7',
 'tokenIn': '0x390f3595bCa2Df7d23783dFd126427CCeb997BF4',
 'minAmountIn': '98287057405581344768',
 'minAmountInUSD': '99983801',
 'steps': [{'stepType': 'swap',
   'dex': '0xV2',
   'tokenOut': '0xA0b86991c6218b36c1d19D4a2e9Eb0cE3606eB48',
   'amountOut': '53033331',
   'tokenIn': '0xdAC17F958D2ee523a2206206994597C13D831ec7',
   'minAmountIn': '0',
   'payload': {'blockNumber': '22163013',
    'buyAmount': '53044454',
    'buyToken': '0xdac17f958d2ee523a2206206994597c13d831ec7',
    'fees': {'integratorFee': None, 'zeroExFee': None, 'gasFee': None},
    'issues

In [ ]:
DestinationStates